# Imports and env loading

In [1]:
import os
import ast
import pandas as pd
from dotenv import load_dotenv

load_dotenv(dotenv_path='/scratch/bhanushali.ar/netflix_semantic/.env')

TMDB_PATH = os.getenv('TMDB_PATH')
MOVIELENS_PATH = os.getenv('MOVIELENS_PATH')

print(TMDB_PATH)
print(MOVIELENS_PATH)

/home/bhanushali.ar/.cache/kagglehub/datasets/tmdb/tmdb-movie-metadata/versions/2
/home/bhanushali.ar/.cache/kagglehub/datasets/sriharshabsprasad/movielens-dataset-100k-ratings/versions/1


# Data preprocessing

In [2]:
movies = pd.read_csv(f"{TMDB_PATH}/tmdb_5000_movies.csv")
credits = pd.read_csv(f"{TMDB_PATH}/tmdb_5000_credits.csv")

credits = credits.rename(columns={'movie_id': 'id'})
df = movies.merge(credits[['id', 'cast', 'crew']], on='id')

print(df.shape)

(4803, 22)


In [3]:
def extract_names(json_str, key='name', limit=None):
    try:
        items = ast.literal_eval(json_str)
        names = [i[key] for i in items if key in i]
        return names[:limit] if limit else names
    except:
        return []

def extract_director(crew_str):
    try:
        crew = ast.literal_eval(crew_str)
        for member in crew:
            if member.get('job') == 'Director':
                return member['name']
    except:
        pass
    return ''

df['genres_list']   = df['genres'].apply(extract_names)
df['keywords_list'] = df['keywords'].apply(extract_names)
df['cast_top5']     = df['cast'].apply(lambda x: extract_names(x, limit=5))
df['director']      = df['crew'].apply(extract_director)

print(df[['title', 'genres_list', 'cast_top5', 'director']].head(3))

                                      title  \
0                                    Avatar   
1  Pirates of the Caribbean: At World's End   
2                                   Spectre   

                                     genres_list  \
0  [Action, Adventure, Fantasy, Science Fiction]   
1                   [Adventure, Fantasy, Action]   
2                     [Action, Adventure, Crime]   

                                           cast_top5        director  
0  [Sam Worthington, Zoe Saldana, Sigourney Weave...   James Cameron  
1  [Johnny Depp, Orlando Bloom, Keira Knightley, ...  Gore Verbinski  
2  [Daniel Craig, Christoph Waltz, Léa Seydoux, R...      Sam Mendes  


In [4]:
def build_text(row):
    parts = [
        row['title'],
        row['overview'] if pd.notna(row['overview']) else '',
        'Genres: ' + ', '.join(row['genres_list']),
        'Keywords: ' + ', '.join(row['keywords_list']),
        'Cast: ' + ', '.join(row['cast_top5']),
        'Director: ' + row['director'],
    ]
    return ' | '.join(p for p in parts if p.strip())

df['rich_text'] = df.apply(build_text, axis=1)

print(df['rich_text'].iloc[0])

Avatar | In the 22nd century, a paraplegic Marine is dispatched to the moon Pandora on a unique mission, but becomes torn between following orders and protecting an alien civilization. | Genres: Action, Adventure, Fantasy, Science Fiction | Keywords: culture clash, future, space war, space colony, society, space travel, futuristic, romance, space, alien, tribe, alien planet, cgi, marine, soldier, battle, love affair, anti war, power relations, mind and soul, 3d | Cast: Sam Worthington, Zoe Saldana, Sigourney Weaver, Stephen Lang, Michelle Rodriguez | Director: James Cameron


In [5]:
df = df.dropna(subset=['overview'])
df = df[df['overview'].str.strip() != '']
df = df[df['vote_count'] >= 10]
df = df.reset_index(drop=True)

print(df.shape)

(4391, 27)


In [6]:
keep = ['id', 'title', 'overview', 'genres_list', 'keywords_list',
        'cast_top5', 'director', 'vote_average', 'vote_count',
        'release_date', 'rich_text']

os.makedirs('/scratch/bhanushali.ar/netflix_semantic/data/processed', exist_ok=True)
df[keep].to_csv('/scratch/bhanushali.ar/netflix_semantic/data/processed/movies_processed.csv', index=False)
print('saved')

saved


In [7]:
ML_PATH = f"{MOVIELENS_PATH}/ml-latest-small"

ratings = pd.read_csv(f"{ML_PATH}/ratings.csv")
links   = pd.read_csv(f"{ML_PATH}/links.csv")

ratings.to_csv('/scratch/bhanushali.ar/netflix_semantic/data/processed/ml_ratings.csv', index=False)
links.to_csv('/scratch/bhanushali.ar/netflix_semantic/data/processed/ml_links.csv', index=False)

print(ratings.shape, links.shape)

(100836, 4) (9742, 3)


In [8]:
sample = df[df['title'] == 'Inception'].iloc[0]
print(sample['rich_text'])

Inception | Cobb, a skilled thief who commits corporate espionage by infiltrating the subconscious of his targets is offered a chance to regain his old life as payment for a task considered to be impossible: "inception", the implantation of another person's idea into a target's subconscious. | Genres: Action, Thriller, Science Fiction, Mystery, Adventure | Keywords: loss of lover, dream, kidnapping, sleep, subconsciousness, heist, redemption, female hero | Cast: Leonardo DiCaprio, Joseph Gordon-Levitt, Ellen Page, Tom Hardy, Ken Watanabe | Director: Christopher Nolan
